# Practice — KNN on `mpg.csv`

Same six stages as the Titanic notebook, on your own.

**Question:** given a car's specifications, was it built in the USA, Europe or Japan?

Target: `origin` &nbsp;·&nbsp; File: `mpg.csv`

In [1]:
import pandas as pd

df = pd.read_csv('mpg.csv')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## 1. Look at the data

> **Flow:** Shape, columns, what is missing.

Run `.info()` and `.shape`. Which column has missing values, and how many?

In [2]:
print(df.shape)
df.info()

#Horsepower has 6 missing values


(398, 9)
<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB


How many cars from each `origin`? Use `value_counts()`.

In [3]:

df['origin'].value_counts()

origin
usa       249
japan      79
europe     70
Name: count, dtype: int64

---
## 2. Stage 1 — Data Cleaning

> **Flow:** Fix missing values, drop unusable columns.

Two jobs:

1. `horsepower` has blanks. Fill them with the median.
2. Drop `name`. In one line below, say why it cannot help the model.

In [4]:

df['horsepower']=df['horsepower'].fillna(df['horsepower'].median())
df.drop(columns=['name'],inplace=True)

#Name doesnt matter for origin


*Why `name` cannot help:*

---
## 3. Features and Target

> **Flow:** `X` is everything the model looks at. `y` is the answer.

In [5]:
x = df.drop(columns=['origin'])
y = df['origin']
print(x.head(1))
print(y.head(1))


    mpg  cylinders  displacement  horsepower  weight  acceleration  model_year
0  18.0          8         307.0       130.0    3504          12.0          70
0    usa
Name: origin, dtype: str


---
## 4. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

Use `test_size=0.2`, `random_state=0`, `stratify=y`.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0, stratify=y)


In [8]:
print(x_test.shape)
print(x_train.shape)

(80, 7)
(318, 7)


---
## 5. Stage 3 — Feature Engineering

> **Flow:** Put every column on the same scale.

**No encoding needed here.** After dropping `name`, every feature is already a number,
so there is no text column left for `OneHotEncoder`. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [9]:
print(x_train.min())
print(x_train.max())


mpg                9.0
cylinders          3.0
displacement      68.0
horsepower        46.0
weight          1613.0
acceleration       8.0
model_year        70.0
dtype: float64
mpg               46.6
cylinders          8.0
displacement     455.0
horsepower       225.0
weight          5140.0
acceleration      24.8
model_year        82.0
dtype: float64


Now scale. `StandardScaler` — `fit_transform` on train, `transform` on test.

In [10]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

---
## 6. Stages 4 and 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

**First without scaling.** Train `KNeighborsClassifier()` on the unscaled data and print the accuracy.

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [13]:
model = KNeighborsClassifier()
model.fit(x_train,y_train)
y_pre = model.predict(x_test)
print(y_pre)

['usa' 'usa' 'usa' 'usa' 'usa' 'usa' 'usa' 'europe' 'usa' 'europe' 'usa'
 'usa' 'europe' 'europe' 'usa' 'usa' 'usa' 'usa' 'japan' 'usa' 'usa'
 'europe' 'usa' 'usa' 'europe' 'japan' 'usa' 'europe' 'usa' 'usa' 'europe'
 'europe' 'japan' 'japan' 'europe' 'usa' 'usa' 'usa' 'usa' 'usa' 'usa'
 'usa' 'japan' 'europe' 'usa' 'usa' 'usa' 'europe' 'usa' 'usa' 'usa' 'usa'
 'usa' 'usa' 'usa' 'usa' 'usa' 'usa' 'usa' 'europe' 'usa' 'europe' 'usa'
 'europe' 'japan' 'usa' 'usa' 'usa' 'usa' 'japan' 'japan' 'europe' 'usa'
 'usa' 'usa' 'japan' 'usa' 'japan' 'usa' 'usa']


**Now with scaling.** Same model, same `k`, scaled data.

In [14]:
from sklearn.preprocessing import StandardScaler

In [15]:
scaler=StandardScaler()
x_train_scaled1=scaler.fit_transform(x_train)
x_test_scaled1=scaler.transform(x_test)


In [16]:
scaled_model=KNeighborsClassifier()
scaled_model.fit(x_train_scaled,y_train)
y_pre_scaled=scaled_model.predict(x_test_scaled)


---
## 7. Stage 6 — Compare

> **Flow:** Two numbers, one difference.

Print both accuracies together. Which is higher, and by how much?

In [17]:

print("before scaling",accuracy_score(y_test,y_pre))
print("After Scaling ",accuracy_score(y_test,y_pre_scaled))

before scaling 0.6875
After Scaling  0.75


*What changed between the two runs:*

---
## 8. Choosing k

> **Flow:** `k` is a dial. Try a few settings and look.

Run a loop over `k = 1, 3, 5, 7, 9, 11, 15, 21` on the **scaled** data.
Print `k` and its accuracy on each line.

In [20]:

for k in [3,5,7,11,13,15,21,25,29]:
    m=KNeighborsClassifier(n_neighbors=k)    
    m.fit(x_train_scaled,y_train)
    m_pre=m.predict(x_test_scaled)
    print(k,round(accuracy_score(y_test,m_pre),2))

3 0.75
5 0.75
7 0.75
11 0.72
13 0.71
15 0.74
21 0.72
25 0.74
29 0.72


*Best k:* &nbsp;&nbsp; *Its accuracy:*

Does the accuracy change a lot across k, or stay roughly flat?

*Your answers:*